# Stage 2 — Agent pipeline

How the second stage works, end to end, before any LLM is plugged in. The code lives in `agent/`; this notebook shows it in use.

Run order: `kyc.ipynb` → `preprocessing.ipynb` → `xgboost.ipynb` → this notebook.

This notebook covers:
1. One flagged case, as a reviewer sees it
2. What each tool returns for that case
3. The pipeline with the no-LLM baseline reviewer
4. Plugging in an LLM

**Leakage rules, enforced in code and tested in `agent/test_tools.py`** (`python -m pytest agent -q`):
- Tools never load `is_fraud`, and the simulator's `fraud_` prefix is stripped from merchant names.
- Each case's tools are tied to that customer and that moment: they take no customer or date arguments, and return only strictly earlier transactions.
- Labels live only in `scores_{split}.csv`, which the evaluation reads; reviewers never see them.

In [1]:
import json

from agent.data import TransactionStore
from agent.pipeline import evaluate, load_flagged, load_thresholds, run
from agent.prompt import SYSTEM_PROMPT, render_case
from agent.reviewers import ScoreThresholdReviewer
from agent.schemas import Case
from agent.tools import TOOL_DEFINITIONS, CaseTools

store = TransactionStore.load()
thresholds = load_thresholds()
flagged = load_flagged("val")
print(f"Thresholds: standalone {thresholds['standalone']:.3f}, flagging {thresholds['flag']:.3f} "
      f"(catches {thresholds['recall_target']:.0%} of validation fraud)")
print(f"Flagged validation cases: {len(flagged):,}")

Thresholds: standalone 0.959, flagging 0.751 (catches 90% of validation fraud)
Flagged validation cases: 2,237


## 1. One flagged case

This is everything a reviewer is given up front: no label and no card number. The customer is identified only by their KYC record ID.

In [2]:
row = flagged.sort_values("xgb_score").iloc[len(flagged) // 2]  # a mid-scoring case, not an obvious one
case = Case(
    case_id=row.trans_num,
    customer_id=store.customer_id(row.cc_num),
    time=row.trans_date_trans_time,
    amount=row.amt,
    merchant=row.merchant,
    category=row.category,
    xgb_score=row.xgb_score,
)
tools = CaseTools(store, row.cc_num, row.trans_date_trans_time)
print(render_case(case))

Flagged transaction:
{
  "case_id": "b80fcc4855e00ce334ea9aea94105348",
  "customer_id": "KYC-00074",
  "time": "2020-05-03T23:13:51",
  "amount": 658.94,
  "merchant": "Metz-Boehm",
  "category": "shopping_pos",
  "xgb_score": 0.9864
}


## 2. What each tool returns

These are the exact outputs an LLM would receive for this case.

In [3]:
print(json.dumps(tools.call("get_kyc_profile"), indent=2))

{
  "customer_id": "KYC-00074",
  "account_open_date": "2016-01-17",
  "onboarding_channel": "Online",
  "residency_status": "Citizen",
  "employment_status": "Employed",
  "source_of_funds": "Salary",
  "annual_income": 91000,
  "income_band": "60\u2013100k",
  "declared_monthly_spend": 3000,
  "declared_txn_per_month": 36,
  "credit_score": 630,
  "credit_limit": 10500,
  "num_products": 1,
  "id_verification_method": "Database check",
  "address_verified": true,
  "phone_verified": true,
  "email_verified": true,
  "pep_flag": false,
  "sanctions_screening": "Clear",
  "kyc_risk_points": 1,
  "kyc_risk_rating": "Low",
  "last_kyc_review_date": "2018-03-23",
  "kyc_review_overdue": false
}


In [4]:
print(json.dumps(tools.call("get_transaction_history", {"limit": 10}), indent=2))

{
  "transactions": [
    {
      "time": "2020-04-30T02:58:34",
      "amount": 50.06,
      "merchant": "Huels-Nolan",
      "category": "gas_transport"
    },
    {
      "time": "2020-04-30T16:50:49",
      "amount": 207.64,
      "merchant": "Kemmer-Buckridge",
      "category": "misc_pos"
    },
    {
      "time": "2020-04-30T17:12:00",
      "amount": 38.28,
      "merchant": "Wuckert-Goldner",
      "category": "home"
    },
    {
      "time": "2020-04-30T19:34:28",
      "amount": 33.27,
      "merchant": "Schoen-Quigley",
      "category": "kids_pets"
    },
    {
      "time": "2020-05-01T09:09:13",
      "amount": 60.59,
      "merchant": "Olson, Becker and Koch",
      "category": "gas_transport"
    },
    {
      "time": "2020-05-01T13:35:11",
      "amount": 22.47,
      "merchant": "Zemlak, Tillman and Cremin",
      "category": "personal_care"
    },
    {
      "time": "2020-05-02T04:01:23",
      "amount": 70.7,
      "merchant": "Cummings LLC",
      "category": 

In [5]:
print(json.dumps(tools.call("get_spending_summary"), indent=2))

{
  "earlier_transactions": 1378,
  "first_transaction": "2019-01-01",
  "amount_overall": {
    "median": 33.89,
    "p95": 150.48,
    "max": 1727.69
  },
  "amount_by_category": {
    "entertainment": {
      "count": 61,
      "median": 34.0,
      "p95": 115.34
    },
    "food_dining": {
      "count": 86,
      "median": 8.77,
      "p95": 89.55
    },
    "gas_transport": {
      "count": 212,
      "median": 55.93,
      "p95": 85.95
    },
    "grocery_net": {
      "count": 33,
      "median": 48.79,
      "p95": 90.33
    },
    "grocery_pos": {
      "count": 137,
      "median": 97.22,
      "p95": 142.55
    },
    "health_fitness": {
      "count": 69,
      "median": 33.51,
      "p95": 146.86
    },
    "home": {
      "count": 121,
      "median": 36.71,
      "p95": 215.66
    },
    "kids_pets": {
      "count": 134,
      "median": 33.79,
      "p95": 123.25
    },
    "misc_net": {
      "count": 61,
      "median": 7.03,
      "p95": 379.86
    },
    "misc_pos"

## 3. Pipeline run with the baseline reviewer

`ScoreThresholdReviewer` confirms a flag when the XGBoost score clears the standalone threshold, so **XGBoost + baseline reviewer must reproduce XGBoost alone exactly**. That checks the plumbing (case loading, tools, run log, scaling of sampled runs) before an LLM is involved.

It runs on a random sample of 500 flagged validation cases, the way the LLM runs will. Runs are written to `data/agent_runs/` and can be resumed.

In [6]:
reviewer = ScoreThresholdReviewer(thresholds["standalone"])
run_path = run("val", reviewer, sample=500, run_name="demo-baseline", store=store)
evaluate("val", run_path).round(3)

val: 500 cases in this run, 0 already done, 500 to review -> C:\Users\regan\OneDrive\Desktop\Uni\Y4S1\BT4012\BT4102-Group-Project\data\agent_runs\demo-baseline_val.jsonl
  100 / 500
  200 / 500
  300 / 500
  400 / 500
  500 / 500
val: 500 cases reviewed of 2,236 flagged (scale x4.47); 0 errors; 1,538 frauds in the split


,alerts,frauds caught,false alarms,precision,recall
system,,,,,
Stage 1 flags (ceiling for any reviewer),2236,1384,852,0.619,0.900
"A. XGBoost alone (all transactions, exact)",1360,1228,132,0.903,0.798
"A. XGBoost alone (reviewed sample, scaled)",1333,1203,130,0.903,0.782
B. XGBoost + score-threshold-0.959 (scaled),1333,1203,130,0.903,0.782


Rows A (sample, scaled) and B should match each other exactly, and be close to A (exact). The gap between them is sampling error: a 500-case sample estimates the full flagged set. The **Stage 1 flags** row is the ceiling: frauds below the flagging threshold never reach the reviewer, so no reviewer can catch them.

## 4. The LLM reviewer

`agent/llm.py` runs the agent loop over any OpenAI-compatible API (OpenAI, Gemini, OpenRouter, local Ollama), so the provider and model are settings. It needs `pip install openai` and the provider's API key. Start with a 20-case pilot to measure tokens and cost per case:

```python
from agent.llm import HISTORY_TOOLS, LLMReviewer, OpenAICompatibleBackend

backend = OpenAICompatibleBackend("openrouter", "<model name>")
run_path = run("val", LLMReviewer(backend), sample=20, store=store)          # system B
# LLMReviewer(backend, include_score=False)                                 # system C: score withheld
# LLMReviewer(backend, allowed_tools=HISTORY_TOOLS)                         # system D: no KYC tool
evaluate("val", run_path)
```

This is the prompt and the tools every LLM reviewer gets:

In [7]:
print(SYSTEM_PROMPT)
print()
print([d["name"] for d in TOOL_DEFINITIONS])

You are a fraud analyst at a card issuer. A scoring model has flagged a card transaction as possibly fraudulent. Investigate it with the tools, then decide whether to raise a fraud alert or clear the flag.

The tools are already limited to this customer and to times before the flagged transaction. Do not pass a customer id, card number, or date. Call them exactly like this:
- get_kyc_profile with {}
- get_spending_summary with {}
- get_transaction_history with {"limit": 20}

Call those tools before you decide. Compare the flagged amount, category, and time of day with the tool results. Most flagged transactions are legitimate. Raise an alert only when the transaction, or the activity just before it, is clearly out of character.

Every number in evidence must appear in a tool result or in the flagged transaction. If a tool returns an error, say that the lookup failed. Do not invent income, merchants, or counts.

When you have decided, answer with a single JSON object and nothing else:
{